# Deep Unfolding for Power Allocation in UAV Communication Systems

This notebook implements a **Deep Unfolding** (algorithm unrolling) approach for power allocation
in a multi-UAV communication system. Deep unfolding translates iterative optimization algorithms
into trainable neural network layers, combining the interpretability of model-based methods with
the performance of data-driven approaches.

## System Model
- **M** UAVs (UAV Station Controllers, USCs), each with **N** antennas
- **K** single-antenna ground users
- **Zero-Forcing (ZF) beamforming** at the transmitter side
- Goal: Allocate transmit power $p_k$ to each user $k$ to maximize the **weighted sum-rate**

## Optimization Problem
$$\max_{\mathbf{p}} \sum_{k=1}^{K} \alpha_k \cdot W \log_2\left(1 + \beta_k p_k\right)$$
$$\text{s.t.} \quad \sum_{k=1}^{K} p_k \leq P_{\text{total}}, \quad p_k \geq 0$$

## Deep Unfolding Approach
We unroll **T iterations** of the Projected Gradient Ascent (PGA) algorithm into a neural network,
where the step sizes and scaling parameters of each iteration are **learned** from data.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.io import loadmat
import os
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ─── System Parameters ───────────────────────────────────────────────────────
M          = 40      # Number of UAVs
N          = 20      # Antennas per UAV
Wband      = 20e6    # Bandwidth [Hz]
P_total_mW = 200     # Total power budget [mW]
P_total_W  = P_total_mW / 1000   # [W]
Noise_var  = 290 * 1.38e-23 * Wband * 10 ** (9 / 10)  # Thermal noise

user_counts   = [10, 15, 20, 25, 30, 35]  # K values to evaluate
no_Monte      = 200   # Monte Carlo samples for evaluation
no_train      = 5000  # Training samples per K value
no_val        = 500   # Validation samples

# ─── Deep Unfolding Hyper-parameters ─────────────────────────────────────────
NUM_LAYERS    = 10    # Number of unfolded layers (iterations)
LR            = 1e-3  # Learning rate
EPOCHS        = 200   # Training epochs
BATCH_SIZE    = 128   # Mini-batch size

# ── Numerical constants ─────────────────────────────────────────────────────
# Pre-computed once to avoid repeated tensor creation inside forward passes
LN2    = torch.log(torch.tensor(2.0))  # used for log-base-2 gradient computations
EPS    = 1e-6                          # small positive value for gate/alpha clamping
EPS_30 = 1e-30                         # near-zero guard inside log arguments

print(f'System: M={M} UAVs, N={N} antennas, P_total={P_total_mW} mW')
print(f'Deep Unfolding: T={NUM_LAYERS} layers, LR={LR}, epochs={EPOCHS}')

## Channel Generation
If the `.mat` files are available, we load the pre-computed path-loss data.
Otherwise, we generate synthetic large-scale fading coefficients from a
log-normal distribution, which is standard in cell-free massive MIMO literature.

In [ ]:
def generate_beta_hat(K, batch_size=1, beta_mean_dB=28, beta_std_dB=8):
    """
    Generate effective channel gain vectors beta_hat of shape (batch_size, K).

    Each beta_hat[k] represents the normalized effective channel power for user k
    after ZF beamforming.  We model it as log-normal (dB-domain Gaussian).

    Parameters
    ----------
    K            : number of users
    batch_size   : number of channel realisations
    beta_mean_dB : mean SNR in dB  (typical for UAV-to-ground ~500 m, ~28 dB)
    beta_std_dB  : standard deviation of shadow fading in dB

    Returns
    -------
    beta_hat : ndarray, shape (batch_size, K), linear-scale channel gains
    """
    beta_dB = np.random.randn(batch_size, K) * beta_std_dB + beta_mean_dB
    # Note: 0.1 sigma below adds small log-normal jitter to model
    # channel estimation error / small-scale fading variation.
    beta_lin = 10 ** (beta_dB / 10)
    return beta_lin.astype(np.float32)


def load_beta_from_mat(K, no_samples):
    """
    Load beta_hat from pre-computed .mat files if available.
    Falls back to synthetic generation otherwise.
    """
    try:
        channel_data    = loadmat('channel1.mat')
        uav_user_path   = loadmat('uavuserpath1.mat')['UAV_User_path']
        beta_mk_full    = uav_user_path / Noise_var          # shape (M_file, K_file)
        beta_mk_K       = beta_mk_full[:min(M, beta_mk_full.shape[0]),
                                        :min(K, beta_mk_full.shape[1])]
        beta_hat_base   = np.mean(beta_mk_K[:K, :K], axis=0)  # shape (K,)

        # Bootstrap samples by adding small log-normal perturbations
        noise = np.random.lognormal(0, 0.1, size=(no_samples, K)).astype(np.float32)
        beta_batch = (beta_hat_base * noise).astype(np.float32)
        print(f'  Loaded real channel data for K={K}.')
        return beta_batch
    except FileNotFoundError:
        return generate_beta_hat(K, batch_size=no_samples)


# Quick sanity check
beta_sample = generate_beta_hat(K=10, batch_size=4)
print('beta_hat sample shape:', beta_sample.shape)
print('beta_hat sample (dB SNR):', 10 * np.log10(beta_sample[0]))

## Utility Functions

- **Simplex projection**: projects a vector onto the probability simplex scaled by $P_{\text{total}}$
  (kept for the convergence comparison analysis)
- **Softmax allocation**: converts unconstrained $\theta$ to a valid power allocation
- **Rate computation**: computes per-user rates and weighted sum-rate / min-rate
- **Alpha weights**: computes the proportional-fairness weights $\alpha_k = \beta_k / \sum_j \beta_j$

In [ ]:
# ─── Projection onto scaled simplex ─────────────────────────────────────────
def simplex_project_torch(v, P_max):
    """
    Project each row of v (shape: batch x K) onto
    {p : p >= 0, sum(p) = P_max}  (Euclidean projection onto scaled simplex).

    Algorithm: Duchi et al. (2008) O(K log K).
    """
    batch, K = v.shape
    u = torch.sort(v, dim=-1, descending=True).values   # sort descending
    cssv = torch.cumsum(u, dim=-1)                        # cumulative sum
    rho_idx = torch.arange(1, K + 1, device=v.device, dtype=v.dtype)
    cond = u - (cssv - P_max) / rho_idx > 0              # bool mask
    # rho = last index (1-based) satisfying the condition
    rho = cond.sum(dim=-1, keepdim=True).clamp(min=1)    # (batch, 1)
    theta = (torch.gather(cssv, -1, rho - 1) - P_max) / rho.float()
    return torch.clamp(v - theta, min=0.0)


def compute_rates(beta, p, Wband=Wband):
    """
    Compute per-user rates [bps] for given beta (batch x K) and p (batch x K).
    Uses the simplified model: R_k = W * log2(1 + beta_k * p_k)
    """
    return Wband * torch.log2(1 + beta * p)


def weighted_sum_rate(beta, p, alpha):
    """Weighted sum-rate [bps].  alpha, beta, p all shape (batch, K)."""
    rates = compute_rates(beta, p)
    return (alpha * rates).sum(dim=-1)   # (batch,)


def min_rate(beta, p):
    """Min per-user rate [bps]."""
    return compute_rates(beta, p).min(dim=-1).values  # (batch,)


def compute_alpha(beta):
    """Proportional-fairness weights:  alpha_k = beta_k / sum_j beta_j."""
    return beta / (beta.sum(dim=-1, keepdim=True) + EPS_30)


# Quick test
beta_t  = torch.tensor([[1e-8, 2e-8, 5e-9]], dtype=torch.float32)
p_test  = simplex_project_torch(torch.rand_like(beta_t), P_total_W)
alpha_t = compute_alpha(beta_t)
print('p_test sum:', p_test.sum().item(), '(<= P_total_W =', P_total_W, ')')
print('alpha_t:', alpha_t)

## Deep Unfolding Network Architecture

We unroll the **Natural Gradient Ascent** algorithm in the **log-domain** (softmax space).

### Parameterisation
Instead of working directly in the power-simplex $\{\mathbf{p}\geq 0, \mathbf{1}^\top\mathbf{p}=P_{\text{total}}\}$,
we use unconstrained log-domain variables $\boldsymbol{\theta}\in\mathbb{R}^K$ and recover powers via:

$$p_k = P_{\text{total}} \cdot \frac{e^{\theta_k}}{\sum_j e^{\theta_j}}$$

This **automatically** satisfies $p_k > 0$ and $\sum_k p_k = P_{\text{total}}$ for any $\boldsymbol{\theta}$.

### One layer update (natural gradient in $\boldsymbol{\theta}$)

$$\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} + \mathbf{\Lambda}^{(t)} \odot \nabla_{\boldsymbol{\theta}} f(\boldsymbol{\theta}^{(t)})$$

where the natural gradient is:
$$[\nabla_{\boldsymbol{\theta}} f]_k = \frac{p_k}{P_{\text{total}}} \left(g_k - \langle g \rangle_p\right),
\quad g_k = \frac{\alpha_k \beta_k}{\ln 2 \cdot (1 + \beta_k p_k)}, \quad
\langle g \rangle_p = \sum_j \frac{p_j}{P_{\text{total}}} g_j$$

and $\mathbf{\Lambda}^{(t)}$ is a **learnable per-user step-size matrix**.

### Advantages over simplex-projected PGA
- ✅ Always positive powers — no degenerate corner solutions
- ✅ Numerically stable — no catastrophic cancellation in projection
- ✅ Unconstrained optimisation in $\theta$ — standard gradient tools apply
- ✅ All powers sum exactly to $P_{\text{total}}$ by construction

In [ ]:
class PGALayer(nn.Module):
    """
    One Deep Unfolding layer using a **softmax-based** (log-domain) power
    parameterisation.

    Instead of working in the power-simplex {p >= 0, sum(p) = P_max} with an
    explicit Euclidean projection, we parameterise the allocation as

        p = P_max * softmax(theta)

    and perform gradient ascent in the unconstrained log-domain variable theta.
    This avoids catastrophic floating-point cancellation and corner solutions
    that plague simplex-projected PGA for this problem.

    Learnable parameters per layer
    ------------------------------
    log_step : scalar log step-size (exp ensures positivity)
    gate     : per-user scaling of the natural gradient (allows different
               effective step sizes across users)
    """

    def __init__(self, K_max: int):
        super().__init__()
        # Initialise log_step near log(0.1) so the first iteration moves
        # the allocation meaningfully without overshooting.
        self.log_step = nn.Parameter(torch.tensor(-1.0))
        # Per-user gate — start at 1 (equal), learned during training
        self.gate = nn.Parameter(torch.ones(K_max))

    def forward(self, theta, beta, alpha, P_max, K):
        """
        Parameters
        ----------
        theta : log-domain allocation variable  (batch, K)  unconstrained
        beta  : effective channel gains          (batch, K)
        alpha : user priority weights            (batch, K)
        P_max : total power budget  [scalar, W]
        K     : number of active users

        Returns
        -------
        theta_new : updated log-domain variable  (batch, K)
        """
        step = torch.exp(self.log_step)                     # scalar > 0
        gate = torch.relu(self.gate[:K]) + EPS             # (K,) >= 0

        # Power allocation from current theta
        p = P_max * torch.softmax(theta, dim=-1)           # (batch, K) > 0

        # Gradient of sum_k alpha_k * log2(1 + beta_k * p_k) w.r.t. theta_k
        # Using the chain rule through softmax:
        #   d(WSR)/d(theta_k) = p_k/P_max * (g_k - <g>_p)
        # where g_k = alpha_k * beta_k / (ln2 * (1 + beta_k * p_k))
        #       <g>_p = sum_j (p_j/P_max) * g_j  (p-weighted average)
        g = alpha * beta / (LN2 * (1.0 + beta * p) + EPS_30)
        g_avg = (g * p / P_max).sum(dim=-1, keepdim=True)  # (batch, 1)
        nat_grad = (p / P_max) * (g - g_avg)               # natural gradient (batch, K)

        # Gradient ascent step with per-user gating
        theta_new = theta + step * gate * nat_grad

        return theta_new


class DeepUnfoldingNet(nn.Module):
    """
    Deep Unfolding Network for Power Allocation (softmax parameterisation).

    Architecture: T stacked PGALayer modules operating in log-domain.
    The output power allocation is obtained by applying softmax to the final
    log-domain variable theta.

    Parameters
    ----------
    num_layers : number of unfolded iterations (depth T)
    K_max      : maximum number of users supported
    P_max      : power budget [W]
    """

    def __init__(self, num_layers: int, K_max: int, P_max: float):
        super().__init__()
        self.num_layers = num_layers
        self.K_max      = K_max
        self.P_max      = P_max
        self.layers     = nn.ModuleList(
            [PGALayer(K_max) for _ in range(num_layers)]
        )

    def forward(self, beta, alpha=None):
        """
        Parameters
        ----------
        beta  : effective channel gains  (batch, K)  torch.Tensor
        alpha : user weights             (batch, K)  if None, uses compute_alpha

        Returns
        -------
        p : allocated powers  (batch, K)  summing exactly to P_max, all positive
        """
        K = beta.shape[-1]

        if alpha is None:
            alpha = compute_alpha(beta)

        # Initialise theta at equal power allocation (log-domain)
        theta = torch.zeros_like(beta)   # zeros → softmax gives 1/K per user (uniform start)

        for layer in self.layers:
            theta = layer(theta, beta, alpha, self.P_max, K)

        # Convert log-domain variable to power allocation
        p = self.P_max * torch.softmax(theta, dim=-1)
        return p

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate and inspect
K_max = max(user_counts)
model = DeepUnfoldingNet(num_layers=NUM_LAYERS, K_max=K_max, P_max=P_total_W).to(device)
print(model)
print(f'Total trainable parameters: {model.count_parameters()}')

# Quick sanity check
beta_t = torch.tensor([[100., 500., 1000., 200., 800.]])
p_out  = model(beta_t)
print(f'Output power sum: {p_out.sum().item():.6f} W  (should be {P_total_W})')
print(f'Output min power: {p_out.min().item():.8f} W  (should be > 0)')
print(f'Output powers: {p_out.detach().numpy()}')

## Training

We train the network in an **unsupervised** (self-supervised) manner by directly maximizing
the **weighted sum-rate** objective.  No labelled optimal solutions are needed.

The loss function is:
$$\mathcal{L}(\theta) = -\frac{1}{|\mathcal{B}|} \sum_{b \in \mathcal{B}}
\sum_{k=1}^{K} \alpha_k^{(b)} \cdot W \log_2\!\left(1 + \beta_k^{(b)} p_k^{(b)}\right)$$

We normalise the loss by the bandwidth $W$ for numerical stability.

In [ ]:
def train_model(model, K, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
                n_train=no_train, n_val=no_val, verbose=True):
    """
    Train the DeepUnfoldingNet for a fixed number of users K.

    Returns
    -------
    train_losses : list of mean training losses per epoch (Mbps, negated)
    val_wsr      : list of mean validation weighted-sum-rate per epoch (Mbps)
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # ── Generate data sets ────────────────────────────────────────────────
    beta_train_np = load_beta_from_mat(K, n_train)
    beta_val_np   = load_beta_from_mat(K, n_val)

    beta_train = torch.tensor(beta_train_np, device=device)
    beta_val   = torch.tensor(beta_val_np,   device=device)

    train_losses, val_wsrs = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        # Shuffle
        perm = torch.randperm(n_train, device=device)
        beta_train = beta_train[perm]

        epoch_loss = 0.0
        num_batches = 0

        for start in range(0, n_train, batch_size):
            beta_b = beta_train[start: start + batch_size]
            alpha_b = compute_alpha(beta_b)

            p_out = model(beta_b, alpha_b)

            # Loss = negative mean weighted-sum-rate (normalised to Mbps)
            wsr   = weighted_sum_rate(beta_b, p_out, alpha_b) / 1e6
            loss  = -wsr.mean()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        scheduler.step()

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            alpha_val = compute_alpha(beta_val)
            p_val     = model(beta_val, alpha_val)
            val_wsr   = weighted_sum_rate(beta_val, p_val, alpha_val).mean().item() / 1e6

        train_losses.append(-epoch_loss / num_batches)
        val_wsrs.append(val_wsr)

        if verbose and epoch % 50 == 0:
            print(f'  Epoch {epoch:4d}/{epochs}  |  '
                  f'Train WSR: {train_losses[-1]:.4f} Mbps  |  '
                  f'Val WSR: {val_wsr:.4f} Mbps')

    return train_losses, val_wsrs


print('Training function defined.')

## Baseline Power Allocation Methods

We compare the deep unfolding network against:

| Method | Description |
|--------|-------------|
| **MCUC** | Multi-Criteria Decision for User-Centric – convex problem solved by CVXPY |
| **MMWR** | Maximin Worst Rate – maximises the minimum per-user rate via CVXPY |
| **EPA**  | Equal Power Allocation – $p_k = P / K$ for all users |
| **DU-PGA** | Deep Unfolding PGA (proposed) |

In [ ]:
def mcuc_allocation(beta_hat_np, alpha_np, P_mW=P_total_mW, W=Wband):
    """
    Case 1: Multi-Criteria Decision for User-Centric (MCUC).
    Solves:  max  sum_k alpha_k * W * log2(1 + beta_k * p_k)
             s.t. sum p_k <= P_total, p_k >= 0
    via CVXPY (closed-form water-filling exists but CVXPY handles the general case).
    """
    K   = len(beta_hat_np)
    P_W = P_mW / 1000
    p   = cp.Variable(K)
    obj = cp.Maximize(
        cp.sum(cp.multiply(alpha_np,
                           W * cp.log(1 + cp.multiply(beta_hat_np, p)) / np.log(2)))
    )
    prob = cp.Problem(obj, [cp.sum(p) <= P_W, p >= 0])
    # CLARABEL is more reliable than SCS for this problem class
    try:
        prob.solve(solver=cp.CLARABEL, verbose=False)
    except Exception:
        prob.solve(verbose=False)
    if p.value is None:
        return np.full(K, P_W / K)
    return np.maximum(p.value, 0)


def mmwr_allocation(beta_hat_np, P_mW=P_total_mW, W=Wband):
    """
    Case 2: Maximin Worst Rate (MMWR).
    Solves:  max min_k [W * log2(1 + beta_k * p_k)]
             s.t. sum p_k <= P_total, p_k >= 0
    """
    K      = len(beta_hat_np)
    P_W    = P_mW / 1000
    p      = cp.Variable(K)
    R_min  = cp.Variable()
    constr = [p >= 0, cp.sum(p) <= P_W]
    for k in range(K):
        constr.append(R_min <= beta_hat_np[k] * p[k])
    prob = cp.Problem(cp.Maximize(R_min), constr)
    # CLARABEL is more reliable than SCS for this problem class
    try:
        prob.solve(solver=cp.CLARABEL, verbose=False)
    except Exception:
        prob.solve(verbose=False)
    if p.value is None:
        return np.full(K, P_W / K)
    return np.maximum(p.value, 0)


def epa_allocation(K, P_mW=P_total_mW):
    """Case 3: Equal Power Allocation."""
    return np.full(K, P_mW / (1000 * K), dtype=np.float32)


def compute_rates_np(beta_np, p_np, W=Wband):
    """Compute per-user rates [bps] using numpy."""
    return W * np.log2(1 + beta_np * p_np)


print('Baseline methods defined.')

## Train the Deep Unfolding Model

We train a **single** model on all user counts simultaneously
(the network handles variable K thanks to the `K_max` parameter).

In [ ]:
# Train on the largest K so the gate weights cover all user indices.
# At test time we simply use the first K entries.
K_train = max(user_counts)   # K = 35

print(f'Training Deep Unfolding Network on K={K_train} users ...')
train_losses, val_wsrs = train_model(
    model, K=K_train, epochs=EPOCHS, verbose=True
)

# ── Training curve ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train WSR (Mbps)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weighted Sum-Rate (Mbps)')
axes[0].set_title('Training Weighted Sum-Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_wsrs, color='orange', label='Val WSR (Mbps)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Weighted Sum-Rate (Mbps)')
axes[1].set_title('Validation Weighted Sum-Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('DU_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final validation WSR: {val_wsrs[-1]:.4f} Mbps')

## Evaluation: Monte Carlo Comparison

We evaluate all methods over `no_Monte` independent channel realisations for each
user count $K \in \{10, 15, 20, 25, 30, 35\}$ and report:

1. **Average worst-user rate** (min-rate) [Mbps]  — fairness metric
2. **Average sum-rate** [Mbps]  — throughput metric

In [ ]:
results = {method: {'worst': [], 'sum': [], 'wsum': []}
           for method in ['MCUC', 'MMWR', 'EPA', 'DU-PGA']}

for K in user_counts:
    beta_eval_np = generate_beta_hat(K, batch_size=no_Monte)   # (no_Monte, K)
    print(f'\nEvaluating K={K} ...')

    worst_rates = {m: np.zeros(no_Monte) for m in results}
    sum_rates   = {m: np.zeros(no_Monte) for m in results}
    wsum_rates  = {m: np.zeros(no_Monte) for m in results}

    for trial in range(no_Monte):
        beta_np  = beta_eval_np[trial]                 # (K,)
        alpha_np = beta_np / (beta_np.sum() + 1e-30)   # proportional-fairness

        # ── MCUC ─────────────────────────────────────────────────────────
        p_mcuc  = mcuc_allocation(beta_np, alpha_np)
        r_mcuc  = compute_rates_np(beta_np, p_mcuc)
        worst_rates['MCUC'][trial]  = r_mcuc.min() / 1e6
        sum_rates  ['MCUC'][trial]  = r_mcuc.sum() / 1e6
        wsum_rates ['MCUC'][trial]  = (alpha_np * r_mcuc).sum() / 1e6

        # ── MMWR ─────────────────────────────────────────────────────────
        p_mmwr  = mmwr_allocation(beta_np)
        r_mmwr  = compute_rates_np(beta_np, p_mmwr)
        worst_rates['MMWR'][trial]  = r_mmwr.min() / 1e6
        sum_rates  ['MMWR'][trial]  = r_mmwr.sum() / 1e6
        wsum_rates ['MMWR'][trial]  = (alpha_np * r_mmwr).sum() / 1e6

        # ── EPA ──────────────────────────────────────────────────────────
        p_epa   = epa_allocation(K)
        r_epa   = compute_rates_np(beta_np, p_epa)
        worst_rates['EPA'][trial]   = r_epa.min() / 1e6
        sum_rates  ['EPA'][trial]   = r_epa.sum() / 1e6
        wsum_rates ['EPA'][trial]   = (alpha_np * r_epa).sum() / 1e6

        # ── DU-PGA ───────────────────────────────────────────────────────
        beta_t  = torch.tensor(beta_np[None, :], device=device)
        alpha_t = torch.tensor(alpha_np[None, :], device=device)
        with torch.no_grad():
            p_du = model(beta_t, alpha_t).squeeze().cpu().numpy()
        r_du  = compute_rates_np(beta_np, p_du)
        worst_rates['DU-PGA'][trial]  = r_du.min() / 1e6
        sum_rates  ['DU-PGA'][trial]  = r_du.sum() / 1e6
        wsum_rates ['DU-PGA'][trial]  = (alpha_np * r_du).sum() / 1e6

    for m in results:
        results[m]['worst'].append(worst_rates[m].mean())
        results[m]['sum'  ].append(sum_rates  [m].mean())
        results[m]['wsum' ].append(wsum_rates [m].mean())
        print(f'  {m:<10s}  worst={results[m]["worst"][-1]:.4f} Mbps' +
              f'  sum={results[m]["sum"][-1]:.4f} Mbps' +
              f'  wsum={results[m]["wsum"][-1]:.4f} Mbps')

print('\nEvaluation complete.')


## Results: Average Worst-User Rate and Sum-Rate

In [ ]:
COLORS  = {'MCUC': 'tab:blue', 'MMWR': 'tab:orange',
           'EPA': 'tab:green',  'DU-PGA': 'tab:red'}
MARKERS = {'MCUC': 'o', 'MMWR': 's', 'EPA': '^'  , 'DU-PGA': 'D'}
LABELS  = {'MCUC': 'MCUC (CVXPY)', 'MMWR': 'MMWR (CVXPY)',
           'EPA': 'EPA', 'DU-PGA': 'DU-PGA (proposed)'}

x = np.array(user_counts)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Average Worst-User Rate
ax = axes[0]
for m in ['MCUC', 'MMWR', 'EPA', 'DU-PGA']:
    ax.plot(x, results[m]['worst'], marker=MARKERS[m],
            color=COLORS[m], linewidth=2, markersize=7, label=LABELS[m])
ax.set_xlabel('Number of Users K', fontsize=12)
ax.set_ylabel('Average Worst-User Rate (Mbps)', fontsize=12)
ax.set_title('Worst-User Rate vs. K', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Average (Unweighted) Sum-Rate
ax = axes[1]
for m in ['MCUC', 'MMWR', 'EPA', 'DU-PGA']:
    ax.plot(x, results[m]['sum'], marker=MARKERS[m],
            color=COLORS[m], linewidth=2, markersize=7, label=LABELS[m])
ax.set_xlabel('Number of Users K', fontsize=12)
ax.set_ylabel('Average Sum-Rate (Mbps)', fontsize=12)
ax.set_title('Unweighted Sum-Rate vs. K', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Average Weighted Sum-Rate (objective that MCUC & DU both optimise)
ax = axes[2]
for m in ['MCUC', 'MMWR', 'EPA', 'DU-PGA']:
    ax.plot(x, results[m]['wsum'], marker=MARKERS[m],
            color=COLORS[m], linewidth=2, markersize=7, label=LABELS[m])
ax.set_xlabel('Number of Users K', fontsize=12)
ax.set_ylabel('Average Weighted Sum-Rate (Mbps)', fontsize=12)
ax.set_title('Weighted Sum-Rate vs. K\n($\\alpha_k = \\beta_k / \\sum\\beta_j$)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Deep Unfolding vs. Baselines: Power Allocation Performance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('DU_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved to DU_comparison.png')


## Convergence Analysis: Deep Unfolding vs. Fixed-Step Natural Gradient

We show how the weighted sum-rate evolves layer-by-layer for the deep unfolding network
compared to a natural gradient ascent with a fixed step size — demonstrating the benefit
of **learned** step sizes.

In [ ]:
def natural_grad_fixed_step(beta, alpha, P_max, num_iters=NUM_LAYERS, step=0.5):
    """
    Natural gradient ascent in log-domain (softmax parameterisation) with a fixed step.
    Returns WSR at each iteration for a single sample.
    """
    K     = len(beta)
    theta = np.zeros(K)
    wsr   = []
    for _ in range(num_iters):
        p      = P_max * np.exp(theta) / np.exp(theta).sum()
        wsr.append((alpha * Wband * np.log2(1 + beta * p)).sum() / 1e6)
        g      = alpha * beta / (np.log(2) * (1 + beta * p) + 1e-30)
        g_avg  = (g * p / P_max).sum()
        nat_g  = (p / P_max) * (g - g_avg)
        theta  = theta + step * nat_g
    return wsr


def du_layer_by_layer(model, beta_t, alpha_t):
    """
    Run DU forward pass and record WSR after each layer.
    """
    K      = beta_t.shape[-1]
    theta  = torch.zeros_like(beta_t)
    wsr    = []
    with torch.no_grad():
        for layer in model.layers:
            theta = layer(theta, beta_t, alpha_t, model.P_max, K)
            p_t   = model.P_max * torch.softmax(theta, dim=-1)
            wsr.append(
                weighted_sum_rate(beta_t, p_t, alpha_t).item() / 1e6
            )
    return wsr


# Use a single random sample for illustration
K_demo   = 20
beta_np  = generate_beta_hat(K_demo, batch_size=1)[0]       # (K,)
alpha_np = beta_np / beta_np.sum()

beta_t  = torch.tensor(beta_np[None, :], device=device)
alpha_t = torch.tensor(alpha_np[None, :], device=device)

# Fixed-step natural gradient (step tuned heuristically)
fixed_wsr  = natural_grad_fixed_step(beta_np, alpha_np, P_total_W, step=1.0)
du_layer_wsr = du_layer_by_layer(model, beta_t, alpha_t)

# EPA reference
p_epa_np  = np.full(K_demo, P_total_W / K_demo)
wsr_epa_ref = (alpha_np * Wband * np.log2(1 + beta_np * p_epa_np)).sum() / 1e6

plt.figure(figsize=(8, 5))
plt.axhline(wsr_epa_ref, color='tab:green', linestyle='--', linewidth=1.5,
            label=f'EPA ({wsr_epa_ref:.1f} Mbps)')
plt.plot(range(1, NUM_LAYERS + 1), fixed_wsr, marker='o',
         label='Natural Gradient (fixed step)', color='tab:orange', linewidth=1.8)
plt.plot(range(1, NUM_LAYERS + 1), du_layer_wsr, marker='D',
         label='Deep Unfolding (learned)', color='tab:red', linewidth=2.5)
plt.xlabel('Layer / Iteration', fontsize=12)
plt.ylabel('Weighted Sum-Rate (Mbps)', fontsize=12)
plt.title(f'Convergence per Layer  (K={K_demo})', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('DU_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Convergence plot saved to DU_convergence.png')


## Learned Parameters Analysis

We visualise the learnable step sizes and per-user gate weights across the $T$ layers,
providing insight into what the network has learned.

In [ ]:
step_sizes = []
gate_norms = []

for i, layer in enumerate(model.layers):
    step_sizes.append(torch.exp(layer.log_step).item())
    gate_norms.append(layer.gate[:K_train].detach().cpu().numpy())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Step sizes
axes[0].bar(range(1, NUM_LAYERS + 1), step_sizes, color='steelblue')
axes[0].set_xlabel('Layer', fontsize=12)
axes[0].set_ylabel('Step Size (exp(log_step))', fontsize=12)
axes[0].set_title('Learned Step Sizes per Layer', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

# Gate weights heatmap
gate_matrix = np.stack(gate_norms, axis=0)  # (NUM_LAYERS, K_train)
im = axes[1].imshow(gate_matrix, aspect='auto', cmap='viridis',
                    origin='lower')
plt.colorbar(im, ax=axes[1])
axes[1].set_xlabel('User Index', fontsize=12)
axes[1].set_ylabel('Layer', fontsize=12)
axes[1].set_title('Learned Per-User Gate Weights', fontsize=13)

plt.tight_layout()
plt.savefig('DU_parameters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Parameter visualisation saved to DU_parameters.png')

## Computation Time Comparison

A key advantage of the deep unfolding approach is its **runtime efficiency** during inference.
While MCUC and MMWR require solving convex programs (milliseconds to seconds each),
the DU network runs as a simple forward pass through $T=10$ layers.

We measure average inference time per sample.

In [ ]:
import time

K_bench   = 20
n_bench   = 100
beta_bench_np = generate_beta_hat(K_bench, n_bench)
beta_bench_t  = torch.tensor(beta_bench_np, device=device)

# ── Time MCUC ────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
for i in range(n_bench):
    alpha_i = beta_bench_np[i] / beta_bench_np[i].sum()
    mcuc_allocation(beta_bench_np[i], alpha_i)
t_mcuc = (time.perf_counter() - t0) / n_bench * 1000   # ms

# ── Time MMWR ────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
for i in range(n_bench):
    mmwr_allocation(beta_bench_np[i])
t_mmwr = (time.perf_counter() - t0) / n_bench * 1000

# ── Time EPA ─────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
for i in range(n_bench):
    epa_allocation(K_bench)
t_epa = (time.perf_counter() - t0) / n_bench * 1000

# ── Time DU-PGA (batch inference) ────────────────────────────────────────────
model.eval()
t0 = time.perf_counter()
with torch.no_grad():
    _ = model(beta_bench_t[:, :K_bench])
t_du_batch = (time.perf_counter() - t0) / n_bench * 1000   # ms per sample (batched)

# DU sequential (per-sample)
t0 = time.perf_counter()
with torch.no_grad():
    for i in range(n_bench):
        model(beta_bench_t[i:i+1, :K_bench])
t_du_seq = (time.perf_counter() - t0) / n_bench * 1000

methods = ['MCUC\n(CVXPY)', 'MMWR\n(CVXPY)', 'EPA', 'DU-PGA\n(batch)', 'DU-PGA\n(single)']
times   = [t_mcuc, t_mmwr, t_epa, t_du_batch, t_du_seq]

print('Average inference time per sample:')
for m, t in zip(methods, times):
    print(f'  {m.replace(chr(10), " "):<22s} {t:.4f} ms')

plt.figure(figsize=(9, 5))
bars = plt.bar(methods, times,
               color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple'])
plt.ylabel('Avg Inference Time per Sample (ms)', fontsize=12)
plt.title(f'Computational Cost Comparison  (K={K_bench}, n={n_bench} samples)', fontsize=13)
plt.yscale('log')
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
             f'{t:.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('DU_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Trained Model

We save the trained model weights so they can be loaded later without re-training.

In [ ]:
import os

save_path = 'du_pga_model.pth'
torch.save({
    'model_state_dict' : model.state_dict(),
    'num_layers'       : NUM_LAYERS,
    'K_max'            : K_max,
    'P_max_W'          : P_total_W,
    'results'          : results,
    'user_counts'      : user_counts,
}, save_path)
print(f'Model saved to: {os.path.abspath(save_path)}')

# ── How to reload ────────────────────────────────────────────────────────────
checkpoint = torch.load(save_path, map_location=device, weights_only=False)
model_loaded = DeepUnfoldingNet(
    num_layers=checkpoint['num_layers'],
    K_max=checkpoint['K_max'],
    P_max=checkpoint['P_max_W']
).to(device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
model_loaded.eval()
print('Model reloaded successfully.')